# Niche model — vector overlays (03b)

Reads the generalised NiN layer from **`03a_nisjemodell_raster.ipynb`** and applies
the vector overlays:

tørrfall → observation substrate → clip to grunnlinje → subtract land →
**checkpoint (parquet)** → Naturkart attributes → save.

The checkpoint after land subtraction lets you rerun the attribute step (reload
cell) without rerunning the overlays.

In [1]:
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd

import mnk
import mnk.attributes
import mnk.utils
import mnk.vectorize
import mnk.rules

CRS = "EPSG:25833"
EPSG = 25833
NODATA = 255

## Sources and paths

In [2]:
REPO = Path.cwd()

# Inngang: generalisert NiN-lag fra 03a.
nin_vec_path = REPO / "nisjedata-modell_norge_2026_25833_generalized.gpkg"

# Observasjons-substrat (autoritativt, vektor-overlay).
obs_substrat_parquet = (
    "gs://niva-geodata/MarintNaturKart/results/nisjedata-substrat-klassifisering_norge_latest_25833.geo.parquet"
)
# Tørrfall med innelukket vann (prepped av mnk.sources.prepare_torrfall_filled).
torrfall_parquet = "gs://niva-geodata/MarintNaturKart/results/torrfall-filled_norge_25833.geo.parquet"
land_parquet = "gs://niva-geodata/MarintNaturKart/aux/land-mask-norway_FKB-osm-seaborder-subdivided-fullres_v3_25833_20260924.geo.parquet"

# Drivere for usikkerhet (per-akse beslutningsmargin, NKM_nisjedata_kvalitet):
# substratsannsynlighet, Kd490 (lys), dybde (lys), og salinitet.
subprob_path = "gs://niva-geodata/MarintNaturKart/results/3band_probability_calibrated.tif"
kd_path = "gs://niva-geodata/MarintNaturKart/results/KD490_norge_2020-2025_filled_25833.tif"
depth_path = "gs://niva-geodata/MarintNaturKart/features/feature_norge_dem50_depth_filled.tif"
salinity_path = "gs://niva-geodata/MarintNaturKart/input/hi/Norfjords_160_surf_salt_mean_UTM33.tif"

# Checkpoint (kun parquet) etter klipp/land, før attributter.
clipped_checkpoint = REPO / "nisjemodell_norge_2026_25833_clipped.geo.parquet"

## NiN coding

`nin_int` (1-12) is coded first; `nin_kode_fkid`/`NiN_name` are derived **once**
just before the checkpoint (via `set_nin_codes`, two scalar `map`s instead of
`apply`).

In [3]:
# NiN integer (1-12) -> (nin_kode_fkid, NiN_name), fra mnk.rules.
NIN_INT_MAP = {
    1: mnk.rules.classify_nisje("eufotisk", "brakk", "løst", "innenfor"),   # NA-MC04
    2: mnk.rules.classify_nisje("eufotisk", "salt",  "løst", "innenfor"),   # NA-MA04
    3: mnk.rules.classify_nisje("eufotisk", "brakk", "fast", "innenfor"),   # NA-MC01
    4: mnk.rules.classify_nisje("eufotisk", "salt",  "fast", "innenfor"),   # NA-MA01
    5: mnk.rules.classify_nisje("eufotisk", "brakk", "løst", "utenfor"),    # NA-MC04
    6: mnk.rules.classify_nisje("afotisk",  "brakk", "løst", "utenfor"),    # NA-MC04
    7: mnk.rules.classify_nisje("eufotisk", "salt",  "løst", "utenfor"),    # NA-MA05
    8: mnk.rules.classify_nisje("afotisk",  "salt",  "løst", "utenfor"),    # NA-MA06
    9: mnk.rules.classify_nisje("eufotisk", "brakk", "fast", "utenfor"),    # NA-MC02
    10: mnk.rules.classify_nisje("afotisk", "brakk", "fast", "utenfor"),    # NA-MC02
    11: mnk.rules.classify_nisje("eufotisk", "salt", "fast", "utenfor"),    # NA-MA02
    12: mnk.rules.classify_nisje("afotisk", "salt", "fast", "utenfor"),     # NA-MA03
}
NIN_CODE = {k: v[0] for k, v in NIN_INT_MAP.items()}
NIN_NAME = {k: v[1] for k, v in NIN_INT_MAP.items()}

def set_nin_codes(gdf):
    """Derive nin_kode_fkid/NiN_name from nin_int (two vectorised maps)."""
    gdf["nin_kode_fkid"] = gdf["nin_int"].map(NIN_CODE)
    gdf["NiN_name"] = gdf["nin_int"].map(NIN_NAME)
    return gdf

# Only euphotic "outside" codes change inside tørrfall (intertidal/euphotic).
TORRFALL_REMAP = {5: 1, 7: 2, 9: 3, 11: 4}

# Observation overlay swaps ONLY the substrate axis of an existing NiN code.
SUBSTRAT_TO_FAST = {1: 3, 2: 4, 5: 9, 6: 10, 7: 11, 8: 12}
SUBSTRAT_TO_LOST = {v: k for k, v in SUBSTRAT_TO_FAST.items()}

BUNN_TO_INT = {"løsbunn": 0, "blanding": 0, "fastbunn": 1}  # 0=soft, 1=hard

## Load NiN layer (from 03a) + merge slivers

In [ ]:
gdf_nin = gpd.read_file(nin_vec_path)
gdf_nin = gdf_nin.rename(columns={"class_int": "nin_int"})
gdf_nin = gdf_nin[gdf_nin["nin_int"] != NODATA].reset_index(drop=True)
gdf_nin = gdf_nin.explode(index_parts=False).reset_index(drop=True)
if gdf_nin.crs is None or gdf_nin.crs.to_epsg() != EPSG:
    gdf_nin = gdf_nin.set_crs(CRS)


gdf_nin = mnk.vectorize.merge_slivers(gdf_nin, min_area=2500, max_thinness=0.1, class_col="nin_int")

print(f"Polygoner: {len(gdf_nin):,}")
gdf_nin.head()

## Load observation substrate (national)

Substrate for the grid itself comes from the modelled raster in the loop below.
Here only the **observation** polygons (`nisjedata-substrat-klassifisering`) are
loaded; they are applied as a vector overlay after vectorisation to preserve their
geometry.

In [5]:
BUNN_TO_INT = {"løsbunn": 0, "blanding": 0, "fastbunn": 1}  # 0=soft, 1=hard

gdf_obs = gpd.read_parquet(obs_substrat_parquet).to_crs(CRS)
gdf_obs = gdf_obs[gdf_obs["BunnType"].isin(BUNN_TO_INT)].reset_index(drop=True)
gdf_obs["_obs_fast"] = gdf_obs["BunnType"].map(BUNN_TO_INT).astype("uint8")
gdf_obs = gdf_obs[["geometry", "_obs_fast"]]
_ = gdf_obs.sindex  # build spatial index
print(f"Observasjons-polygoner: {len(gdf_obs):,}")

Observasjons-polygoner: 130,919


## Tørrfall in the vector step

The tørrfall polygons are small and detailed. Instead of rasterising them to 50 m,
they are applied as a vector overlay: euphotic NiN polygons overlapping tørrfall are
split, and the parts lying **innenfor** (inside) tørrfall are remapped:

`NA-MA05→NA-MA04`, `NA-MC02→NA-MC01`, `NA-MA02→NA-MA01`, `NA-MC04→NA-MC04`.

The tørrfall polygons do not follow the land mask exactly. The water mask (which
follows the shoreline) is therefore pre-clipped once with tørrfall in
`mnk.sources.prepare_torrfall_filled`: water pieces that become isolated (cut off
from open sea between tørrfall and land) are merged into tørrfall. Open sea is
contiguous and never becomes isolated, so tørrfall never grows seaward.

In [ ]:
# Tørrfall with enclosed water between tørrfall and land merged
gdf_torrfall = gpd.read_parquet(torrfall_parquet).to_crs(CRS)[["geometry"]].reset_index(drop=True)
print(f"Tørrfall-polygoner: {len(gdf_torrfall):,}")

Tørrfall-polygoner: 2,079,804


In [ ]:
remap_mask = gdf_nin["nin_int"].isin(TORRFALL_REMAP)
gdf_remap = gdf_nin[remap_mask].reset_index(drop=True)
gdf_keep = gdf_nin[~remap_mask].reset_index(drop=True)

inside = mnk.vectorize.intersect_layers(gdf_remap, gdf_torrfall)
inside["nin_int"] = inside["nin_int"].map(TORRFALL_REMAP).astype(gdf_remap["nin_int"].dtype)

outside = mnk.vectorize.subtract_layer(gdf_remap, gdf_torrfall)

gdf_nin_tf = pd.concat([gdf_keep, outside, inside], ignore_index=True)
gdf_nin_tf = gpd.GeoDataFrame(gdf_nin_tf, geometry="geometry", crs=CRS)
gdf_nin_tf = gdf_nin_tf[~gdf_nin_tf.is_empty].explode(index_parts=False).reset_index(drop=True)
print(f"Polygoner etter tørrfall-overlay: {len(gdf_nin_tf):,}")

del gdf_nin, gdf_torrfall, gdf_remap, gdf_keep, inside, outside, remap_mask
gc.collect()

/home/kim/work/marint-naturkart-nivaR/mnk/vectorize.py:496: UserWarning: GeoSeries.notna() previously returned False for both missing (None) and empty geometries. Now, it only returns False for missing values. Since the calling GeoSeries contains empty geometries, the result has changed compared to previous versions of GeoPandas.
Given a GeoSeries 's', you can use '~s.is_empty & s.notna()' to get back the old behaviour.

To further ignore this warning, you can do: 
import warnings; warnings.filterwarnings('ignore', 'GeoSeries.notna', UserWarning)
  result = result[~result.geometry.is_empty & result.geometry.notna()]


Polygoner etter tørrfall-overlay: 2,628,192


0

## Observation substrate in the vector step

The observation polygons are authoritative for substrate. They are applied as a
vector overlay so the observed borders are preserved exactly (no pixel snapping):

1. `intersection` with the NiN layer → each piece inherits light+salinity from the
   underlying polygon (large observations split only along light/salinity borders),
2. only the substrate axis is swapped via `SUBSTRAT_TO_FAST` / `SUBSTRAT_TO_LOST`,
3. `difference` stamps the observation footprint out of the NiN layer, and the
   swapped observation pieces are pasted back in,
4. adjacent observation pieces with the same NiN code are dissolved, so the
   observations are not split up unnecessarily.

In [8]:
_dtype = gdf_nin_tf["nin_int"].dtype

# Inherit light+salinity from the underlying NiN polygon; only substrate is swapped
obs_pieces = mnk.vectorize.intersect_layers(gdf_obs, gdf_nin_tf)

/home/kim/work/marint-naturkart-nivaR/mnk/vectorize.py:496: UserWarning: GeoSeries.notna() previously returned False for both missing (None) and empty geometries. Now, it only returns False for missing values. Since the calling GeoSeries contains empty geometries, the result has changed compared to previous versions of GeoPandas.
Given a GeoSeries 's', you can use '~s.is_empty & s.notna()' to get back the old behaviour.

To further ignore this warning, you can do: 
import warnings; warnings.filterwarnings('ignore', 'GeoSeries.notna', UserWarning)
  result = result[~result.geometry.is_empty & result.geometry.notna()]


In [ ]:
obs_pieces["nin_int"] = obs_pieces["nin_int"].astype("int64")
_fast = obs_pieces["_obs_fast"] == 1
obs_pieces.loc[_fast, "nin_int"] = obs_pieces.loc[_fast, "nin_int"].map(lambda v: SUBSTRAT_TO_FAST.get(v, v))
obs_pieces.loc[~_fast, "nin_int"] = obs_pieces.loc[~_fast, "nin_int"].map(lambda v: SUBSTRAT_TO_LOST.get(v, v))
obs_pieces["nin_int"] = obs_pieces["nin_int"].astype(_dtype)

n_before = len(obs_pieces)
obs_pieces = (
    obs_pieces[["nin_int", "geometry"]]
    .dissolve(by="nin_int", as_index=False)
    .explode(index_parts=False)
    .reset_index(drop=True)
)
print(f"Observasjonsbiter: {n_before:,} -> {len(obs_pieces):,} etter sammenslåing")

nin_minus = mnk.vectorize.subtract_layer(gdf_nin_tf, gdf_obs[["geometry"]])

nin_minus["_substrat_kilde"] = mnk.attributes.SOURCE_SUBSTRAT_NIVA
obs_pieces["_substrat_kilde"] = mnk.attributes.SOURCE_SUBSTRAT_NGU

gdf_nin_obs = pd.concat([nin_minus, obs_pieces], ignore_index=True)
gdf_nin_obs = gpd.GeoDataFrame(gdf_nin_obs, geometry="geometry", crs=CRS)
gdf_nin_obs = gdf_nin_obs[~gdf_nin_obs.is_empty].explode(index_parts=False).reset_index(drop=True)
# Derive nin_kode_fkid/NiN_name once
gdf_nin_obs = set_nin_codes(gdf_nin_obs)
gdf_nin_obs = gdf_nin_obs[["nin_int", "nin_kode_fkid", "NiN_name", "_substrat_kilde", "geometry"]]
print(f"Polygoner etter observasjons-overlay: {len(gdf_nin_obs):,}")

del gdf_nin_tf, gdf_obs, obs_pieces, nin_minus
gc.collect()

## Subtract land

## Clip to grunnlinje (1 nautical mile)

Remove polygons outside grunnlinje + 1 nautical mile, before subtracting land.

In [10]:
print(f"Polygoner før grunnlinje-klipp: {len(gdf_nin_obs):,}")
gdf_nin_obs = mnk.vectorize.clip_to_grunnlinje(gdf_nin_obs)
print(f"Polygoner etter grunnlinje-klipp: {len(gdf_nin_obs):,}")

Polygoner før grunnlinje-klipp: 2,846,751


/home/kim/work/marint-naturkart-nivaR/.pixi/envs/default/lib/python3.13/site-packages/osgeo/gdal.py:13117: RuntimeWarning: A geometry of type MULTIPOLYGON is inserted into layer data of geometry type POLYGON, which is not normally allowed by the GeoPackage specification, but the driver will however do it. To create a conformant GeoPackage, if using ogr2ogr, the -nlt option can be used to override the layer geometry type. This warning will no longer be emitted for this combination of layer and feature geometry type.
  return _gdal.wrapper_GDALVectorTranslateDestName(*args)


Polygoner etter grunnlinje-klipp: 2,846,271


In [11]:
land = gpd.read_parquet(land_parquet)
land = land.to_crs(CRS)

gdf_nin_land = mnk.vectorize.subtract_land(gdf_nin_obs, land)
print(f"Polygoner etter land-subtraksjon: {len(gdf_nin_land):,}")

del land, gdf_nin_obs
gc.collect()

Subtracting land from 2,195,292 of 2,846,271 polygons...


Polygoner etter land-subtraksjon: 2,736,485


0

## Checkpoint: clipped layer (parquet)

Saves the clipped layer (nin_int + code/name + substrate source) as
parquet. Run the **reload** cell below to rerun the attribute step without rerunning
the overlays.


In [ ]:
gdf_nin_land.to_parquet(clipped_checkpoint, compression="snappy")
print(f"Checkpoint lagret: {clipped_checkpoint.name} ({len(gdf_nin_land):,} polygoner)")

## Add Naturkart attributes

Derive the data-model fields (nin_flate_NK) available at this point. `nkm_lys`,
`nkm_lm-sa`, `nkm_lm-dk` and `nkm_lm-tv` take their codes from `KODE k_nkm_<..>`
(see `mnk.rules.NKM_*`).

`nkm_usikkerhet` (method `NKM_nisjedata_kvalitet`) is the per-axis
decision-margin estimate from `mnk.utils.nisje_usikkerhet`: a polygon is uncertain
when any axis sits near its class boundary — substrate `|P_soft − P_hard|` (mixture
shrinks the margin), light `|z_eu − depth| / z_eu` (relative), and salinity `|S − 18|`. The
worst (most uncertain) axis wins; NGU-observed substrate forces `lav`.

In [4]:
gdf_nin_land = gpd.read_parquet(clipped_checkpoint)
print(f"Lastet checkpoint: {len(gdf_nin_land):,} polygoner")

Lastet checkpoint: 2,736,485 polygoner


In [5]:
is_obs = (gdf_nin_land["_substrat_kilde"] == mnk.attributes.OBS_SUBSTRAT_KILDE).to_numpy()
usikkerhet = mnk.utils.nisje_usikkerhet(
    gdf_nin_land,
    kd_path=str(kd_path),
    depth_path=str(depth_path),
    salinity_path=str(salinity_path),
    subprob_path=str(subprob_path),
    is_obs=is_obs,
)
gdf_nin_land = mnk.attributes.add_nkm_attributes(gdf_nin_land, usikkerhet=usikkerhet)
print(gdf_nin_land["nkm_usikkerhet"].value_counts())
gdf_nin_land.head()

  nisje_usikkerhet: tile 744/744 (100%), 496,263/2,736,485 polygons covered
nkm_usikkerhet
medium    1206947
lav       1113775
høy        415763
Name: count, dtype: int64


,opphav_kode,nin_int,nin_kode_fkid,NiN_name,nkm_prosess,nkm_lys,nkm_lm-sa,nkm_lm-dk,nkm_lm-tv,metode_klassifisering,nkm_usikkerhet,metode_usikkerhet,datakilde,geometry
0,NKM,8,NA-MA06,Afotisk saltvanns-sedimentbunn,A,y,fgh,0ABCD,LM-TV_0,NKM_regelbasert_dyp,medium,NKM_nisjedata_kvalitet,lys_modellert_niva | salinitet_modellert_hi | ...,"POLYGON ((970024.167 7942253.331, 970025 79422..."
1,NKM,12,NA-MA03,Afotisk fast saltvannsbunn,A,y,fgh,EFGY,LM-TV_0,NKM_regelbasert_dyp,medium,NKM_nisjedata_kvalitet,lys_modellert_niva | salinitet_modellert_hi | ...,"POLYGON ((970100 7942255.093, 970100 7942250, ..."
2,NKM,12,NA-MA03,Afotisk fast saltvannsbunn,A,y,fgh,EFGY,LM-TV_0,NKM_regelbasert_dyp,høy,NKM_nisjedata_kvalitet,lys_modellert_niva | salinitet_modellert_hi | ...,"POLYGON ((970450 7942263.226, 970450 7942250, ..."
3,NKM,12,NA-MA03,Afotisk fast saltvannsbunn,A,y,fgh,EFGY,LM-TV_0,NKM_regelbasert_dyp,høy,NKM_nisjedata_kvalitet,lys_modellert_niva | salinitet_modellert_hi | ...,"POLYGON ((967724.85 7942199.9, 967650 7942150,..."
4,NKM,12,NA-MA03,Afotisk fast saltvannsbunn,A,y,fgh,EFGY,LM-TV_0,NKM_regelbasert_dyp,medium,NKM_nisjedata_kvalitet,lys_modellert_niva | salinitet_modellert_hi | ...,"POLYGON ((968213.739 7942211.261, 968300 79421..."


## Save result

In [6]:
fname = mnk.utils.to_filename("nkm-nisjemodell", "norge", "2026", CRS.split(":")[1])
out_dir = REPO

gdf_nin_land.to_parquet(out_dir / f"{fname}.geo.parquet", compression="snappy")
print(f"Lagret {len(gdf_nin_land):,} polygoner -> {fname}.geo.parquet")

Lagret 2,736,485 polygoner -> nkm-nisjemodell_norge_2026_25833.geo.parquet


In [16]:
# Upload to PostGIS (optional)
mnk.utils.to_postgis(gdf_nin_land, fname)

NIVAGIS_CONNECTION_STR not set. Skipping PostGIS upload.
